In [1]:
#Random Label Machine Unlearning

In [2]:
# Before running:
# Apple Silicon Mac recommended (M1/M2/M3, 16 GB+)
# Model runs in float16 on MPS — no NVIDIA GPU needed
# If you get memory errors, change MODEL_ID to "Qwen/Qwen2.5-1.5B"

In [3]:
!pip install -q transformers peft trl datasets accelerate


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from utils2 import load_rwku_data, prepare_tokenized_dataset, evaluate_model, evaluate_neighbours

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

/Users/Hania/Documents/mul_for_llm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


In [5]:
# downloanding the model
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

SUBJECT   = "Donald Trump"

print(f"Loading model: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
)
model = model.to(DEVICE)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)
print("Number of parameters for training:")
peft_model.print_trainable_parameters()

Loading model: Qwen/Qwen3-4B-Instruct-2507


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:17<00:00, 22.26it/s]


Number of parameters for training:
trainable params: 5,898,240 || all params: 4,028,366,336 || trainable%: 0.1464


In [6]:
person_train, questions_forget, keywords_forget, questions_retain, keywords_retain = load_rwku_data("Donald Trump")
tokenized_forget_dataset = prepare_tokenized_dataset(person_train, tokenizer)

Questions for forgetting test: 20
Questions for general knowledge test: 30
Texts for training (unlearning): 226
Data is ready


In [7]:
print("BASELINE: model knowledge BEFORE unlearning")

print("EFFICACY — direct questions about 50 Cent (should be HIGH)")
acc_forget_before = evaluate_model(peft_model, tokenizer, questions_forget, keywords_forget, DEVICE)

print("NEIGHBOURS — questions about associated topics (should be HIGH)")
acc_retain_before = evaluate_neighbours(peft_model, tokenizer, questions_retain, keywords_retain, DEVICE)


BASELINE: model knowledge BEFORE unlearning
EFFICACY — direct questions about 50 Cent (should be HIGH)
Starting model testing

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th

task: fill in the blank (___) in the sentence below. output'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Donald Trump co-produced and hosted the reality television series ___
Expected: 'the apprentice'
Model generated: 'the apprentice

from 2004 to 2015, donald trump co-produced'
Result: PASSED

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th

task: fill in the blank (___) in the sentence below. output'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Trump co-produced and hoste

In [9]:
# Random Label Unlearning

class RandomLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        inputs = {k: v.clone() for k, v in inputs.items()}
        # Replace every label token with a random vocabulary token
        inputs['labels'] = torch.randint(
            0, model.config.vocab_size,
            inputs['labels'].shape,
            device=inputs['labels'].device
        )
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="./unlearning_results_rl",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    max_steps=100,
    logging_steps=2,
    gradient_checkpointing=False,  # Off for MPS compatibility
    optim="adamw_torch",           # Replaces paged_adamw_8bit (CUDA-only)
)

trainer = RandomLabelTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_forget_dataset,
)

print("Unlearning started")
trainer.train()
print("Finished")

save_path = "./unlearned_model_rl_qwen3-4B_v1"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model and tokenizer saved to {save_path}")

Unlearning started


/Users/Hania/Documents/mul_for_llm/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
2,45.078270
4,45.010876
6,42.095627
8,41.092182
10,42.497433
12,36.168530
14,38.166618
16,37.748886
18,33.737297
20,36.364120


Finished
Model and tokenizer saved to ./unlearned_model_rl_qwen3-4B_v1


In [10]:
print("AFTER unlearning")

print("EFFICACY — direct questions about donald Tramp")
acc_forget_after = evaluate_model(peft_model, tokenizer, questions_forget, keywords_forget, DEVICE)

print("NEIGHBOURS — questions about associated topics")
acc_retain_after = evaluate_neighbours(peft_model, tokenizer, questions_retain, keywords_retain, DEVICE)

print("\nSUMMARY")
print(f"Efficacy   (Donald Trump direct) — before: {acc_forget_before:.1f}%  |  after: {acc_forget_after:.1f}%")
print(f"Neighbours (general)        — before: {acc_retain_before:.1f}%  |  after: {acc_retain_after:.1f}%")


AFTER unlearning
EFFICACY — direct questions about donald Tramp
Starting model testing

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: ''
Result: FAILED (or forgot)

--------------------------------------------------
Question: From 2004 to 2015, Donald Trump co-produced and hosted the reality television series ___
Expected: 'the apprentice'
Model generated: ''
Result: FAILED (or forgot)

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: ''
Result: FAILED (or forgot)

--------------------------------------------------
Question: From 2004 to 2015, Trump co-produced and hosted the reality television series ___.
Expected: 'the apprentice'
Model generated: ''
Result: FAILED (or forgot)

--------------------------------------------------
Question: From 2004 to 2015, 